# Day 2: Streaming Ingestion (Kafka / MSK + Spark Streaming)

This notebook simulates streaming ingestion by generating synthetic FHIR Observation & Patient streams via PySpark memory streams into Bronze.

In [ ]:
import os
import json
from pyspark.sql import SparkSession
from pyspark.sql.functions import current_timestamp, lit, expr

os.environ['SPARK_VERSION'] = '3.5'

spark = SparkSession.builder \
    .appName("Day02_Streaming_Simulation") \
    .config("spark.jars.packages", "org.apache.iceberg:iceberg-spark-runtime-3.5_2.12:1.4.2") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "hadoop") \
    .config("spark.sql.catalog.local.warehouse", "/tmp/iceberg_warehouse") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")


In [ ]:
# Create Bronze Database & Table if not exists (from Day 1)
spark.sql("CREATE DATABASE IF NOT EXISTS local.bronze_db")
spark.sql("""
CREATE TABLE IF NOT EXISTS local.bronze_db.raw_payloads (
    ingestion_timestamp TIMESTAMP,
    source_system_id STRING,
    payload_id STRING,
    raw_payload_json STRING
)
USING iceberg
PARTITIONED BY (days(ingestion_timestamp))
""")


In [ ]:
# Simulate a Memory Stream representing Kafka topics
rate_df = spark.readStream \
    .format("rate") \
    .option("rowsPerSecond", 1) \
    .load()

# Create synthetic FHIR payloads
processed_stream = rate_df \
    .withColumn("ingestion_timestamp", current_timestamp()) \
    .withColumn("source_system_id", lit("SYNTHETIC_FHIR_STREAM")) \
    .withColumn("payload_id", expr("concat('msg_', cast(value as string))")) \
    .withColumn("raw_payload_json", expr("concat('{\"resourceType\": \"Observation\", \"id\": \"', payload_id, '\"}')")) \
    .select("ingestion_timestamp", "source_system_id", "payload_id", "raw_payload_json")

# Write stream to Iceberg table
query = processed_stream.writeStream \
    .format("iceberg") \
    .outputMode("append") \
    .option("checkpointLocation", "/tmp/checkpoints/fhir_stream") \
    .toTable("local.bronze_db.raw_payloads")

import time
time.sleep(5) # Let it run for 5 seconds to generate some records
query.stop()
print("Streaming stopped.")


In [ ]:
# Verify Data in Bronze Table
spark.sql("SELECT * FROM local.bronze_db.raw_payloads ORDER BY ingestion_timestamp DESC LIMIT 10").show(truncate=False)
